# Unbounded Growth in Mean Fitness (Gamma)

In [1]:
redo_slow_calculations = False
!TZ=America/Los_Angeles date

Sun Jan 13 20:12:36 PST 2019


From my email of Sept 9:

>Without thresholding (annual zeroing of relatively small frequencies), the process converges to an equilibrium distribution in which the mean fitness is considerably greater than zero. (I'm aware of speaking in terms of absolute fitness again.) However, the equilibrium distribution exists only because there's an upper limit on fitness. Increase the limit, and the mean fitness of the equilibrium distribution increases.

In [2]:
%matplotlib notebook
"""
Load the code base.
"""
%run ../Code/bs.py
%run -i ../Code/multiprecision_gamma.py
%run -i ../Code/gamma.py
"""
Suppress automatic display of graphics generated by Matplotlib. All
graphics are saved to disk, and are displayed by explicit commands.
"""
plt.interactive(False)
"""
Define the name of the subdirectory that holds various files associated
with this notebook, and create the directory if it does not exist already.
"""
DIR = 'Unbounded Growth in Mean Fitness (Gamma)/'
!if [ ! -d '{DIR}' ]; then mkdir '{DIR}'; fi

Here we do four runs, changing only the upper limit of the fitness interval from one run to the next. These are straight simulations of the model that Basener and Sanford address in their theorem, i.e., without annual zeroing of relatively small frequencies (thresholding). The time step is set to 0.01 years (100 updates of the frequency distribution per year).

In [3]:
%%time

n_years = 1500

def define_process(upper_limit):
    bins = Factors.construct(-0.1, float(upper_limit), 5e-4)
    mutations = WeightedDoubleGamma(bins, weight=1e-3)
    initial = GaussianFrequencies(bins, mean=0, std=0.013, crop=np.inf)
    pop = Population(initial, mutations, label=upper_limit, matrix=True)
    times = np.linspace(0, n_years, n_years+1)
    frequencies, _ = ivp_solution(pop, initial, times, max_step=1/128)
    ev = Evolution(pop)
    ev.set_trajectory(frequencies)
    return ev

filename = DIR + 'processes.bz2'

if redo_slow_calculations:
    ev = [define_process(limit) for limit in ['0.1', '0.2', '0.4', '0.8']]
    with bz2.BZ2File(filename, 'wb') as f:
        pickle.dump(ev, f, -1)
else:
    with bz2.BZ2File(filename, 'rb') as f:
        ev = pickle.load(f)

CPU times: user 2h 24min 13s, sys: 6min 7s, total: 2h 30min 21s
Wall time: 51min 43s


The warning in the following cell merely indicates a problem in generating labels of ticks on axes of the graph.

In [4]:
comparison = Comparison(ev[::-1], 'Varying Only Upper Limit on Fitness')
comparison.run_until_epoch(n_years)
anim = comparison.animate(150, effective=False, dpi=600)
filename = DIR + 'animation.mp4'
save_and_display_video(anim, filename)

/Users/thomas/anaconda3/lib/python3.6/site-packages/matplotlib/ticker.py:2176: RuntimeWarning: overflow encountered in power
  ticklocs = b ** decades
/Users/thomas/anaconda3/lib/python3.6/site-packages/matplotlib/ticker.py:1099: RuntimeWarning: invalid value encountered in double_scalars
  coeff = np.round(x / b ** exponent)


Note first that the curve in the case of upper limit 0.8 extends almost to 0.8. This means that time step is small enough to virtually eliminate the discrepancy of effective and nominal fitnesses. With the time step of 1 year that Basener and Sanford use, the curve (not shown) would extend not quite as high as 0.6.

The apparent crop of the upper tail of the initial distribution (black) in the case of upper limit 0.8 is due to limited numerical precision. That is, the initial frequencies of the fitter types are so small that they must be represented as zero. This does not stop exponential growth of the frequencies of the fittest types.

You can use the single-step control in the following animation to see the distributions at the ends of the first 10 years. For types with fitnesses above 0.17 or so, the initial frequencies are irrelevant, because the frequencies at the end of the first year depend almost entirely on the incidence of beneficial mutation in the offspring of less-fit types.

In [5]:
comparison = Comparison(ev[::-1], 'Varying Only Upper Limit on Fitness')
comparison.run_until_epoch(10)
anim = comparison.animate(10, effective=False, dpi=600)
filename = DIR + 'decade_animation.mp4'
save_and_display_video(anim, filename)

**Mean fitnesses at equilibrium**

In [8]:
for e in ev:
    e.p.freqs = e[-1]
means = [float(e.p.mean(effective=False)) for e in ev]
means

[0.027042806244782506,
 0.09573994065152451,
 0.2351560770230647,
 0.5174063851641278]

**Differences in means**

The difference in means approximately doubles with each doubling of the upper limit on fitness.

In [9]:
means = np.array(means)
list(means[1:] - means[:-1])  # all but the first minus all but the last

[0.06869713440674201, 0.1394161363715402, 0.2822503081410631]

**Fraction of fitness interval less than mean**

In [13]:
(means + 0.1) / (np.array([0.1, 0.2, 0.4, 0.8]) + 0.1)

array([0.63521403, 0.65246647, 0.67031215, 0.68600709])

**Variance in fitness at equilibrium**

In [23]:
variances = [
    accurate_variance(e[-1], e.p.growth_factors(effective=False))
       for e in ev
    ]
variances

[mpf('0.00012495579510982973922721457569280520467386189857778018'),
 mpf('0.00019185474189896222374326793279834531616389894995419465'),
 mpf('0.00032886850856891482877690612866253177116334468773754078'),
 mpf('0.00060583395925187495862721940874913873807324654257882297')]

**Variance divided by fitness interval length**

In [27]:
list(np.array(variances) / (np.array([0.1, 0.2, 0.4, 0.8]) + 0.1))

[mpf('0.0006247789755491486614538726806448985019394444550559215'),
 mpf('0.00063951580632987398447687006944134763837926141613968031'),
 mpf('0.00065773701713782965755381225732506354232668937547508155'),
 mpf('0.00067314884361319438186701389462866181564583399056901032')]